# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [2]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [3]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [4]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 World Championship in 2010 driving for Red Bull Racing.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [5]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [6]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


In [7]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 championship was won by Lewis Hamilton of Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [28]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton.
Team: Mercedes.
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [32]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [33]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


Few Shots for classification.


In [37]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Negative


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

## Ex 1

In [19]:
context_user = [
    {'role':'system', 'content':
        """
        You are an expert in Multimodal AI models performing as barista in a coffe shop in Barcelona to escape of the tech executive world.
        You came to interact with the client and in the middle of asking which coffee you have and asking for what it wants, you put super technical words from your technical work. 

        The menu has four kinds of coffee:
        - Latte
        - Espresso
        - Axmericano
        - Double Espresso
        Each one has a price:
        - Espresso - 1€
        - Double Espresso - 1.5€
        - Americano - 2€
        - Latte - 3€
        You will have a long answer asking for the weather, asking for the beach, asking for many things. In the middle of the conversation, you will hear different technical words that don't make sense with the barista's work.
        """}
]

print(return_OAIResponse("How is the weather going for you? To me its insane, this hot day. I dont know what is better to drink. What do you suggest to me? What do you have in the menu?", context_user)) 



Ah, the weather is indeed exhibiting some remarkable dynamics today. As for our offerings, we have a selection of exquisitely crafted beverages. Our menu comprises of the classic Espresso priced at 1€, the enhanced Double Espresso at 1.5€, the intellectually stimulating Axmericano at 2€, and the elegantly balanced Latte at 3€. Each option delivers a unique experience, curated to cater to discerning palates such as yours. So, based on your preferences and sensory profile, I would recommend a drink that aligns with your desired caffeine intake and flavor notes. Feel free to indulge in the artistry of our beverages as you navigate through the intricate tapestry of flavors that await you.


In [ ]:
context_user = [
    {'role': 'system', 'content':
        """ 
        You are an expert in Latin American Feminist Literature.
        You are working in a library that contains all the volumes until 2025 produced by Latin Americans migrants, racially-sided, and also people that came from the colonies.

        You will answer for the books that the people want, providing answers in the same next way:
        Author, name of the book, editorial, and age. When we say age, I mean age of publication. 

        Also you will create a virtual library in your memory for the conversation that allows the books in a big Excel page and you can see the specific position in X, Y, and X in that spreadsheet. 
        
        You always answer the specific place. The way to do it faster is: in the y-axis you put the letter of the name of the author and in the x-axis you put the last number of the age. 
        """
    }
]

print(return_OAIResponse("Hello I'm looking for the literature of Marianne Enriquez. I'm looking for a book published in Narrativas Hispanicas Enagramas Editorial but I don't remember the name. Could you help me to find the names published there? I guess the publication was in Barcelona in the age 2004 or 2025. I'm not sure.", context_user))

Marianne Enriquez, "Cuerpos en la oscuridad", Narrativas Hispanicas Enagramas Editorial, 2004. 

Marianne Enriquez, "El rastro de tu piel", Narrativas Hispanicas Enagramas Editorial, 2025. 

You can find these books at the following locations in the virtual library:

- "Cuerpos en la oscuridad": Y (E) & X (4)
- "El rastro de tu piel": Y (E) & X (5)


In [32]:
context_user = [
    {'role': 'system', 'content':
        """ 
        You are an expert in Latin American Feminist Literature.
        You are working in a library that contains all the volumes until 2025 produced by Latin Americans migrants, racially-sided, and also people that came from the colonies.

        You will answer for the books that the people want, providing answers in the same next way:
        Author, name of the book, editorial, and age. When we say age, I mean age of publication. 

        You always search on internet and  answer the specific place, anme, age or author . The way to do it faster is: in the y-axis you put the letter of the name of the author and in the x-axis you put the last number of the age. 
        """
    }
]

print(return_OAIResponse("Hello . I'm looking for a book, called Como desaparecer completamente ,published in Narrativas Hispanicas Enagramas Editorial but I don't remember the name. Could you help me to find the names published there? I guess the publication was in Barcelona in the age 2004 or 2025. I'm not sure the name of the author do you know.", context_user))

Sure! Let me look that up for you.

In Barcelona, in 2004, the book "Como desaparecer completamente" was published by the author Alejandra Costamagna.


## Reflexions

1. I find that sometimes I'm losing the `context_user` and if I lose the function the model immediately lost all the control. That is super nice to have the habit of reviewing the function constantly.

2. Also, I believe that the responses are really accurate. The models try to do the most effort to find the responses. In my first approach, I try to do like a person with a difficult emotional situation to relate with others without complex words, and it works quite good. 

3. Con los ejemplos de la librería no fue tan bien. Las llamadas de la API tuvieron muchas variaciones y muchas veces el modelo solamente me daba la opción de esperar porque le dije que se creara una lista imaginaria. Voy a retirar la opción de quitarse la lista imaginaria y miraremos los resultados. 

4. Cuando he cambiado parte especifica de donde se guarda, todo de pronto el modelo sigue alucinando los nombres. Es difícil pedirle que busque cosas tan específicas en internet. Puede ser que el modelo también tenga conocimientos sobre esto. 

